In [ ]:
import os
import warnings
from cellpose import models, io

# Configuration
input_dir = '/workspace/Z-projections'
output_dir = '/workspace/Cellpose/Masks'
os.makedirs(output_dir, exist_ok=True)

# Load models
model_sam = models.CellposeModel(gpu=True)
model_nuc = models.CellposeModel(gpu=True)

# Get all images
files = io.get_image_files(input_dir, mask_filter='_masks')

for f in files:
    img = io.imread(f)
    print(f"Processing: {os.path.basename(f)}")

    cell_masks, flows_sam, _ = model_sam.eval(img[1:], diameter=300)
    nuc_masks, flows_nuc, _ = model_nuc.eval(img[3], diameter=100)

    # Save results
    io.save_masks(img, cell_masks, flows_sam, f, savedir=output_dir, suffix='_cell', tif=True)
    io.save_masks(img, nuc_masks, flows_nuc, f, savedir=output_dir, suffix='_nuc', tif=True)

print("Batch processing complete.")

In [ ]:
import os
import numpy as np
import tifffile as tiff
from cellpose import io
from tqdm import tqdm

input_dir = '/workspace/Z-projections'
mask_dir = '/workspace/Cellpose/Masks'
crop_output_dir = '/workspace/Cellpose/cell_crops'
os.makedirs(crop_output_dir, exist_ok=True)

# Define fixed output dimensions
OUTPUT_H, OUTPUT_W = 1024, 1024

files = io.get_image_files(input_dir, mask_filter='_masks')
master_stack_filename = os.path.join(crop_output_dir, "cell_gallery_1024x1024_6ch.tif")

# 1. & 2. Process cells directly (no need to find max dimensions)
all_cells = []
for f in tqdm(files, desc="Processing Cells"):
    base_name = os.path.splitext(os.path.basename(f))[0]
    img = io.imread(f) # [4, H, W]
    
    # Try loading cell mask, skip file if not found
    mask_path = os.path.join(mask_dir, f"{base_name}_cell.tif")
    if not os.path.exists(mask_path):
        continue
    cell_masks = io.imread(mask_path)
    
    nuc_path = os.path.join(mask_dir, f"{base_name}_nuc.tif")
    nuc_masks = io.imread(nuc_path) if os.path.exists(nuc_path) else np.zeros_like(cell_masks)

    for cell_id in np.unique(cell_masks):
        if cell_id == 0: continue
        
        rows, cols = np.where(cell_masks == cell_id)
        y_min, y_max, x_min, x_max = rows.min(), rows.max(), cols.min(), cols.max()
        
        # Crop and mask
        h, w = y_max - y_min + 1, x_max - x_min + 1
        mask_bool = (cell_masks[y_min:y_max+1, x_min:x_max+1] == cell_id)
        
        # Create 6-channel FIXED SIZE canvas [6, 1024, 1024]
        canvas = np.zeros((6, OUTPUT_H, OUTPUT_W), dtype='uint16')
        
        # Calculate centering offsets, ensuring they don't go negative if cell > 1024
        off_y = max(0, (OUTPUT_H - h) // 2)
        off_x = max(0, (OUTPUT_W - w) // 2)
        
        # Determine how much of the cell actually fits in the canvas
        c_h = min(h, OUTPUT_H)
        c_w = min(w, OUTPUT_W)
        
        # Crop the input to fit within the 1024 boundary if the cell is too big
        crop_y_start = 0 if h <= OUTPUT_H else (h - OUTPUT_H) // 2
        crop_x_start = 0 if w <= OUTPUT_W else (w - OUTPUT_W) // 2
        
        # Fill canvas (4 img channels + cell mask + nuc mask)
        # Background pixels remain 0 due to initialized canvas
        canvas[0:4, off_y:off_y+c_h, off_x:off_x+c_w] = np.where(
            mask_bool[crop_y_start:crop_y_start+c_h, crop_x_start:crop_x_start+c_w],
            img[:, y_min+crop_y_start:y_min+crop_y_start+c_h, x_min+crop_x_start:x_min+crop_x_start+c_w],
            0
        )
        canvas[4, off_y:off_y+c_h, off_x:off_x+c_w] = mask_bool[crop_y_start:crop_y_start+c_h, crop_x_start:crop_x_start+c_w].astype('uint16')
        canvas[5, off_y:off_y+c_h, off_x:off_x+c_w] = np.where(
            mask_bool[crop_y_start:crop_y_start+c_h, crop_x_start:crop_x_start+c_w],
            nuc_masks[y_min+crop_y_start:y_min+crop_y_start+c_h, x_min+crop_x_start:x_min+crop_x_start+c_w],
            0
        )
        
        all_cells.append(canvas)

# 3. Save as a single Hyperstack [Cells, Channels, 1024, 1024]
if all_cells:
    final_stack = np.stack(all_cells, axis=0) # [N, 6, 1024, 1024]
    tiff.imwrite(
        master_stack_filename,
        final_stack,
        imagej=True,
        metadata={'axes': 'ZCYX', 'channels': 6} # Z is cell index, C is 6 channels
    )

print(f"Gallery saved! {len(all_cells)} cells in {master_stack_filename}")